# 🧬 Notebook 11 — BiLSTM + GA All Splits

---

<div style="background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%); padding: 24px 28px; border-radius: 16px; margin: 12px 0; border: 1px solid #e94560;">
<h2 style="color: #e94560; margin: 0 0 8px 0; font-size: 1.4em;">🧬 BiLSTM + GA All Splits</h2>
<h3 style="color: #a8dadc; margin: 0 0 16px 0; font-weight: 400; font-size: 1.05em;">Fase B — Eksperimen Model Deep Learning</h3>
<hr style="border: none; border-top: 1px solid #444; margin: 12px 0;">
<table style="color: #ccc; font-size: 0.9em; border-collapse: collapse; width: 100%;">
<tr><td style="padding: 4px 16px 4px 0; white-space:nowrap;">📍 Studi Kasus</td><td><strong style="color: #fff;">PT. XYZ Banjarmasin, Kalimantan Selatan</strong></td></tr>
<tr><td style="padding: 4px 16px 4px 0;">🎓 Mata Kuliah</td><td><strong style="color: #fff;">Tugas Akhir — ES234733</strong></td></tr>
<tr><td style="padding: 4px 16px 4px 0;">🏛 Institusi</td><td><strong style="color: #fff;">Departemen Sistem Informasi, FTEIC — ITS Surabaya</strong></td></tr>
<tr><td style="padding: 4px 16px 4px 0;">👤 Penulis</td><td><strong style="color: #fff;">Muhammad Iqbal Baiduri Yamani — NRP 5026221103</strong></td></tr>
<tr><td style="padding: 4px 16px 4px 0;">🧑‍🏫 Pembimbing</td><td><strong style="color: #fff;">Edwin Riksakomara, S.Kom., M.T.</strong></td></tr>
<tr><td style="padding: 4px 16px 4px 0;">📅 Tahun</td><td><strong style="color: #fff;">2025</strong></td></tr>
</table>
</div>

## 📌 Tujuan Notebook Ini

Notebook 11 menjalankan **BiLSTM + Genetic Algorithm (GA)** pada semua rasio split (`60:40`, `70:30`, `80:20`, `90:10`) dengan fokus:

- Optimasi hyperparameter BiLSTM berbasis **walk-forward CV**
- Final retrain model terbaik per split
- Evaluasi test set dengan metrik **MAPE (utama), MAE, RMSE, R²**
- Perbandingan objektif terhadap baseline Notebook 10

> Mode notebook ini diset ke **SLIM RUN** agar cepat dulu (lebih ringan untuk iterasi awal).

| # | Tahapan | Keterangan |
|---|---------|------------|
| 1 | Import & Setup | Seed, path, style, dan konfigurasi GA cepat |
| 2 | Load Artifacts | `split_*.npz` dan `wf_cv_*.npz` |
| 3 | Helper + Model BiLSTM | Per-window normalization, metrik, arsitektur, operator GA |
| 4 | GA Optimization | Optimasi hyperparameter BiLSTM per split (dengan progress bar) |
| 5 | Visualisasi | Prediksi vs aktual + perbandingan MAPE NB10 vs NB11 |
| 6 | Save Artifacts | Trial log, metrik terbaik, prediksi, dan model |
| 7 | Checklist | Verifikasi kelengkapan deliverable Pipeline |

## ⚙️ 1. Import Library, Konfigurasi, dan GA Search Space

In [ ]:
import os
import json
import random
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm.auto import tqdm

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf  # type: ignore[import-untyped]
tf.get_logger().setLevel("ERROR")
try:
    tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
except Exception:
    pass
logging.getLogger("tensorflow").setLevel(logging.ERROR)

keras             = tf.keras
Sequential        = keras.Sequential
Input             = keras.layers.Input
Bidirectional     = keras.layers.Bidirectional
LSTM              = keras.layers.LSTM
Dense             = keras.layers.Dense
Dropout           = keras.layers.Dropout
Adam              = keras.optimizers.Adam
ReduceLROnPlateau = keras.callbacks.ReduceLROnPlateau
mixed_precision   = keras.mixed_precision

warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"

# ── Reproducibility ───────────────────────────────────────────
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)

# ── Runtime Setup (lokal CPU) ─────────────────────────────────
gpus = []
GPU_ACTIVE = False
mixed_precision.set_global_policy("float32")

# ── Path Konfigurasi Lokal ────────────────────────────────────
IS_COLAB = False
ROOT_DIR = Path(".").resolve().parent
PATHS = {
    "root": ROOT_DIR,
    "logs": ROOT_DIR / "logs",
    "outputs": ROOT_DIR / "outputs",
    "figures": ROOT_DIR / "outputs" / "figures",
    "models": ROOT_DIR / "outputs" / "models",
    "metrics": ROOT_DIR / "outputs" / "metrics",
    "splits": ROOT_DIR / "outputs" / "splits",
    "cv_folds": ROOT_DIR / "outputs" / "cv_folds",
    "predictions": ROOT_DIR / "results" / "predictions",
    "ga_trials": ROOT_DIR / "results" / "ga_trials",
}
for p in PATHS.values():
    if isinstance(p, Path):
        p.mkdir(parents=True, exist_ok=True)

SPLIT_LABELS = ["60:40", "70:30", "80:20", "90:10"]
K_FOLDS = 5
TAU = 8

# ── Matplotlib Style ─────────────────────────────────────────
ACCENT = "#e94560"
ACCENT2 = "#a8dadc"
plt.rcParams.update({
    "figure.facecolor": "#1a1a2e",
    "axes.facecolor": "#16213e",
    "axes.edgecolor": "#444",
    "axes.labelcolor": "#ccc",
    "xtick.color": "#aaa",
    "ytick.color": "#aaa",
    "text.color": "#ddd",
    "grid.color": "#2a2a4a",
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "axes.grid": True,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ── GA Search Space (SLIM cepat dulu) ───────────────────────
SEARCH_SPACE_BILSTM = {
    "lstm_units": [16, 32],
    "dense_units": [16, 32],
    "dropout": [0.1, 0.25],
    "learning_rate": [0.0005, 0.001],
    "batch_size": [16, 32],
}
PARAM_KEYS = list(SEARCH_SPACE_BILSTM.keys())

# ── GA Config (SLIM) ─────────────────────────────────────────
POP_SIZE = 6
N_GENERATIONS = 3
ELITE_SIZE = 2
TOURNAMENT_K = 3
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.25
FIXED_EPOCHS = 12
FINAL_RETRAIN_EPOCHS = 60

# ── NB10 Baseline untuk initial seed + pembanding ───────────
NB10_BASELINE = {
    "90:10": {"MAPE": 17.25137737506791, "MAE": 6797.334689670139, "RMSE": 8778.744798553205, "R2": -0.06491317870316493},
    "80:20": {"MAPE": 26.551118853326034, "MAE": 7338.768147786458, "RMSE": 9882.701133425231, "R2": -0.1694877846059275},
    "60:40": {"MAPE": 32.42514023237129, "MAE": 7186.217042722902, "RMSE": 9887.794783218169, "R2": -0.03807192550120919},
    "70:30": {"MAPE": 37.67476240998375, "MAE": 7850.354829877337, "RMSE": 10574.29930848989, "R2": -0.12953965628414754},
}

_total_combos = 1
for _v in SEARCH_SPACE_BILSTM.values():
    _total_combos *= len(_v)

print("=" * 84)
print("  SETUP — NOTEBOOK 11 (BILSTM + GA ALL SPLITS) [SLIM RUN]")
print("=" * 84)
print("  Env              : Lokal")
print(f"  TensorFlow       : {tf.__version__}")
print("  GPU terdeteksi   : 0  (CPU mode aktif)")
print(f"  Mixed precision  : {mixed_precision.global_policy().name}")
print(f"  GLOBAL_SEED      : {GLOBAL_SEED}")
print(f"  POP_SIZE         : {POP_SIZE}")
print(f"  N_GENERATIONS    : {N_GENERATIONS}")
print(f"  FIXED_EPOCHS     : {FIXED_EPOCHS}")
print(f"  FINAL_RETRAIN    : {FINAL_RETRAIN_EPOCHS}")
print(f"  Search space     : {len(PARAM_KEYS)} params | {_total_combos} total kombinasi")
print("=" * 84)

## 📂 2. Load Artefak Split dan Walk-Forward CV

In [ ]:
# ── Load split + walk-forward artifacts ──────────────────────
split_artifacts = {}
cv_artifacts = {}

for label in SPLIT_LABELS:
    split_path = PATHS["splits"] / f"split_{label.replace(':', '_')}.npz"
    cv_path = PATHS["cv_folds"] / f"wf_cv_{label.replace(':', '_')}.npz"

    if not split_path.exists():
        raise FileNotFoundError(f"Artefak split tidak ditemukan: {split_path}")
    if not cv_path.exists():
        raise FileNotFoundError(f"Artefak walk-forward tidak ditemukan: {cv_path}")

    split_npz = np.load(split_path, allow_pickle=True)
    cv_npz = np.load(cv_path, allow_pickle=True)

    split_artifacts[label] = {
        "X_train": split_npz["X_train"].astype(np.float32),
        "y_train": split_npz["y_train"].astype(np.float32),
        "X_test": split_npz["X_test"].astype(np.float32),
        "y_test": split_npz["y_test"].astype(np.float32),
    }

    cv_artifacts[label] = {
        "train_end_indices": cv_npz["train_end_indices"].astype(int),
        "val_start_indices": cv_npz["val_start_indices"].astype(int),
        "val_end_indices": cv_npz["val_end_indices"].astype(int),
        "k_folds": int(cv_npz["k_folds"]),
        "initial_train_size": int(cv_npz["initial_train_size"]),
    }

rows = []
for label in SPLIT_LABELS:
    art = split_artifacts[label]
    cv = cv_artifacts[label]
    rows.append({
        "split": label,
        "X_train": str(art["X_train"].shape),
        "y_train": str(art["y_train"].shape),
        "X_test": str(art["X_test"].shape),
        "y_test": str(art["y_test"].shape),
        "k_folds": cv["k_folds"],
        "initial_train_size": cv["initial_train_size"],
    })

display(pd.DataFrame(rows))
print("=" * 72)
print("  ✅ Artefak split dan walk-forward CV berhasil dimuat.")
print("=" * 72)

## 🧱 3. Helper Function: Scaling, Metrics, Model BiLSTM, dan GA Operators

In [ ]:
# ── Per-window normalization ─────────────────────────────────
def normalize_per_window(X: np.ndarray, y: np.ndarray):
    mu = X.mean(axis=1)
    sigma = X.std(axis=1) + 1e-8
    X_n = (X - mu[:, None]) / sigma[:, None]
    y_n = (y - mu) / sigma
    return X_n, y_n, mu, sigma

def inverse_y_per_window(y_n: np.ndarray, mu: np.ndarray, sigma: np.ndarray) -> np.ndarray:
    return y_n * sigma + mu

# ── Metrics ───────────────────────────────────────────────────
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = 1.0 - ss_res / (ss_tot + 1e-12)
    mask = np.abs(y_true) > 1e-8
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) if mask.any() else float("nan")
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}

def build_bilstm_ga(
    input_steps=8,
    lstm_units=32,
    dense_units=16,
    dropout=0.25,
    lr=5e-4,
):
    model = Sequential([
        Input(shape=(input_steps, 1)),
        Bidirectional(LSTM(lstm_units, return_sequences=False)),
        Dense(dense_units, activation="relu"),
        Dropout(dropout),
        Dense(1, dtype="float32"),
    ])
    model.compile(optimizer=Adam(learning_rate=lr, clipnorm=1.0), loss="mse")
    return model

def chromosome_to_cfg(chromosome: list[int]) -> dict:
    return {k: SEARCH_SPACE_BILSTM[k][i] for k, i in zip(PARAM_KEYS, chromosome)}

def random_chromosome(rng: np.random.RandomState) -> list[int]:
    return [rng.randint(0, len(SEARCH_SPACE_BILSTM[k])) for k in PARAM_KEYS]

def nb10_seed_chromosome() -> list[int]:
    seed_vals = {
        "lstm_units": 32,
        "dense_units": 16,
        "dropout": 0.25,
        "learning_rate": 0.0005,
        "batch_size": 16,
    }
    return [SEARCH_SPACE_BILSTM[k].index(v) for k, v in seed_vals.items()]

@tf.function(reduce_retracing=True)
def _predict_fn(model, x):
    return model(x, training=False)

def predict_no_retrace(model, X: np.ndarray) -> np.ndarray:
    return _predict_fn(model, tf.constant(X, dtype=tf.float32)).numpy().reshape(-1)

def _iter_cv_folds(cv: dict):
    k = int(cv["k_folds"])
    for i in range(k):
        end_train = int(cv["train_end_indices"][i])
        start_val = int(cv["val_start_indices"][i])
        end_val = int(cv["val_end_indices"][i])
        tr_idx = np.arange(0, end_train)
        val_idx = np.arange(start_val, end_val)
        yield tr_idx, val_idx

def evaluate_chromosome(chromosome, X_train, y_train, cv, seed):
    cfg = chromosome_to_cfg(chromosome)
    fold_mapes = []
    for fold_idx, (tr_idx, val_idx) in enumerate(_iter_cv_folds(cv)):
        Xtr, ytr = X_train[tr_idx], y_train[tr_idx]
        Xval, yval = X_train[val_idx], y_train[val_idx]

        Xtr_n, ytr_n, _, _ = normalize_per_window(Xtr, ytr)
        Xval_n, _, val_mu, val_sigma = normalize_per_window(Xval, yval)
        Xtr_n = Xtr_n[..., None]
        Xval_n = Xval_n[..., None]

        tf.keras.backend.clear_session()
        tf.random.set_seed(seed + fold_idx)
        model = build_bilstm_ga(
            input_steps=TAU,
            lstm_units=cfg["lstm_units"],
            dense_units=cfg["dense_units"],
            dropout=cfg["dropout"],
            lr=cfg["learning_rate"],
        )
        lr_cb = ReduceLROnPlateau(monitor="loss", factor=0.5, patience=6, min_lr=1e-6, verbose=0)
        model.fit(
            Xtr_n, ytr_n,
            shuffle=False,
            epochs=FIXED_EPOCHS,
            batch_size=cfg["batch_size"],
            callbacks=[lr_cb],
            verbose=0,
        )

        y_pred_n = predict_no_retrace(model, Xval_n)
        y_pred = inverse_y_per_window(y_pred_n, val_mu, val_sigma)
        m = regression_metrics(yval, y_pred)
        fold_mapes.append(m["MAPE"])

    return float(np.mean(fold_mapes))

def tournament_select(population, fitness, k, rng):
    cands = [rng.randint(0, len(population)) for _ in range(k)]
    best = min(cands, key=lambda i: fitness[i])
    return population[best][:]

def crossover(p1, p2, rate, rng):
    if rng.random() < rate:
        pt = rng.randint(1, len(p1))
        return p1[:pt] + p2[pt:], p2[:pt] + p1[pt:]
    return p1[:], p2[:]

def mutate(chromosome, rate, rng):
    child = chromosome[:]
    for i, key in enumerate(PARAM_KEYS):
        if rng.random() < rate:
            child[i] = rng.randint(0, len(SEARCH_SPACE_BILSTM[key]))
    return child

print("✅ Helper functions, arsitektur BiLSTM, dan GA operators siap.")

## 🚀 4. GA Optimization Loop untuk Semua Split

In [ ]:
# ── GA Optimization per split ─────────────────────────────────
all_trial_rows = []
best_configs = {}
best_metrics = {}
best_preds = {}
best_history = {}

for label in SPLIT_LABELS:
    art = split_artifacts[label]
    cv = cv_artifacts[label]
    X_train = art["X_train"]
    y_train = art["y_train"]
    X_test = art["X_test"]
    y_test = art["y_test"]

    rng = np.random.RandomState(GLOBAL_SEED)

    population = [nb10_seed_chromosome()] + [random_chromosome(rng) for _ in range(POP_SIZE - 1)]
    fitness = [float("inf")] * POP_SIZE

    best_chrom_ever = nb10_seed_chromosome()
    best_fitness_ever = float("inf")

    print("\n" + "=" * 84)
    print(f"  GA Split {label}  |  POP={POP_SIZE}  GEN={N_GENERATIONS}  FIXED_EPOCHS={FIXED_EPOCHS}")
    print(f"  NB10 Baseline MAPE = {NB10_BASELINE[label]['MAPE']:.4f}%")
    print("=" * 84)

    gen_bar = tqdm(range(N_GENERATIONS), desc=f"🧬 GA Optimization - {label}", unit="gen")

    for gen in gen_bar:
        ind_bar = tqdm(
            range(POP_SIZE),
            desc=f"  ├─ Gen-{gen + 1:02d}",
            total=POP_SIZE,
            leave=False,
        )
        for idx in ind_bar:
            eval_seed = GLOBAL_SEED + gen * POP_SIZE + idx
            fitness[idx] = evaluate_chromosome(population[idx], X_train, y_train, cv, eval_seed)

            cfg_log = chromosome_to_cfg(population[idx])
            all_trial_rows.append({
                "split": label,
                "generation": gen + 1,
                "individual": idx + 1,
                **cfg_log,
                "cv_mape": fitness[idx],
            })

            if fitness[idx] < best_fitness_ever:
                best_fitness_ever = fitness[idx]
                best_chrom_ever = population[idx][:]

            ind_bar.set_postfix({
                "MAPE": f"{fitness[idx]:.2f}%",
                "best": f"{best_fitness_ever:.2f}%",
            })

        elite_idx = list(np.argsort(fitness)[:ELITE_SIZE])
        new_pop = [population[i][:] for i in elite_idx]

        while len(new_pop) < POP_SIZE:
            p1 = tournament_select(population, fitness, TOURNAMENT_K, rng)
            p2 = tournament_select(population, fitness, TOURNAMENT_K, rng)
            c1, c2 = crossover(p1, p2, CROSSOVER_RATE, rng)
            c1 = mutate(c1, MUTATION_RATE, rng)
            c2 = mutate(c2, MUTATION_RATE, rng)
            new_pop.append(c1)
            if len(new_pop) < POP_SIZE:
                new_pop.append(c2)

        population = new_pop
        gen_bar.set_postfix({"best_MAPE": f"{best_fitness_ever:.2f}%"})

    best_cfg = chromosome_to_cfg(best_chrom_ever)
    best_configs[label] = best_cfg

    print(f"\n  Best config  : {best_cfg}")
    print(f"  Best CV MAPE : {best_fitness_ever:.4f}%")

    X_train_n, y_train_n, _, _ = normalize_per_window(X_train, y_train)
    X_test_n, _, test_mu, test_sigma = normalize_per_window(X_test, y_test)
    X_train_n = X_train_n[..., None]
    X_test_n = X_test_n[..., None]

    tf.keras.backend.clear_session()
    tf.random.set_seed(GLOBAL_SEED)
    final_model = build_bilstm_ga(
        input_steps=TAU,
        lstm_units=best_cfg["lstm_units"],
        dense_units=best_cfg["dense_units"],
        dropout=best_cfg["dropout"],
        lr=best_cfg["learning_rate"],
    )

    lr_cb_final = ReduceLROnPlateau(monitor="loss", factor=0.5, patience=8, min_lr=1e-6, verbose=0)
    hist = final_model.fit(
        X_train_n, y_train_n,
        shuffle=False,
        epochs=FINAL_RETRAIN_EPOCHS,
        batch_size=best_cfg["batch_size"],
        callbacks=[lr_cb_final],
        verbose=0,
    )

    y_pred_n = predict_no_retrace(final_model, X_test_n)
    y_pred = inverse_y_per_window(y_pred_n, test_mu, test_sigma)
    m_test = regression_metrics(y_test, y_pred)

    best_metrics[label] = m_test
    best_preds[label] = {
        "y_true": y_test.astype(np.float32),
        "y_pred": y_pred.astype(np.float32),
    }
    best_history[label] = {
        "loss": [float(v) for v in hist.history["loss"]],
    }

    model_path = PATHS["models"] / f"nb11_bilstm_ga_best_{label.replace(':', '_')}.keras"
    final_model.save(model_path)

    nb10_mape = NB10_BASELINE[label]["MAPE"]
    delta = nb10_mape - m_test["MAPE"]
    arrow = "↓" if delta > 0 else "↑"
    print(f"  Test MAPE    : {m_test['MAPE']:.4f}%  (NB10={nb10_mape:.4f}% | Δ{arrow}{abs(delta):.4f}%)")
    print(f"  MAE          : {m_test['MAE']:.4f}")
    print(f"  RMSE         : {m_test['RMSE']:.4f}")
    print(f"  R²           : {m_test['R2']:.4f}")
    print(f"  Model saved  : {model_path}")

print("\n" + "=" * 84)
print("  ✅ GA Optimization BiLSTM selesai untuk semua split.")
print("=" * 84)

## 📊 5. Visualisasi Prediksi vs Aktual dan Perbandingan MAPE

In [ ]:
# ── Prediksi vs Aktual per split ────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor="#1a1a2e")
axes = axes.flatten()

for ax, label in zip(axes, SPLIT_LABELS):
    yt = best_preds[label]["y_true"]
    yp = best_preds[label]["y_pred"]
    ax.set_facecolor("#16213e")

    ax.plot(yt, color=ACCENT2, linewidth=2.0, label="Aktual")
    ax.plot(yp, color=ACCENT, linewidth=1.8, linestyle="--", label="Prediksi BiLSTM+GA")

    mape_val = best_metrics[label]["MAPE"]
    r2_val = best_metrics[label]["R2"]
    ax.set_title(f"Split {label} | MAPE={mape_val:.3f}%  R²={r2_val:.4f}", color="#ddd", fontsize=10)
    ax.set_xlabel("Index Test", color="#ccc")
    ax.set_ylabel("Grand Total", color="#ccc")
    ax.legend(fontsize=8, labelcolor="#ccc", facecolor="#1a1a2e", edgecolor="#444")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(8))
    for sp in ax.spines.values():
        sp.set_color("#444")

fig.suptitle("Notebook 11 — BiLSTM+GA: Prediksi vs Aktual", color="#e94560", fontsize=13, y=1.01)
plt.tight_layout()
fig_pred_path = PATHS["figures"] / "nb11_bilstm_ga_predictions.png"
fig.savefig(fig_pred_path, dpi=160, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()
print(f"✅ Figur prediksi tersimpan: {fig_pred_path}")

# ── Bar chart: MAPE BiLSTM Baseline (NB10) vs BiLSTM+GA ─────
fig2, ax2 = plt.subplots(figsize=(10, 5), facecolor="#1a1a2e")
ax2.set_facecolor("#16213e")

x = np.arange(len(SPLIT_LABELS))
width = 0.35
mape_nb10 = [NB10_BASELINE[l]["MAPE"] for l in SPLIT_LABELS]
mape_nb11 = [best_metrics[l]["MAPE"] for l in SPLIT_LABELS]

bars1 = ax2.bar(x - width / 2, mape_nb10, width, label="BiLSTM Baseline (NB10)", color=ACCENT2, alpha=0.85)
bars2 = ax2.bar(x + width / 2, mape_nb11, width, label="BiLSTM + GA (NB11)", color=ACCENT, alpha=0.85)

ax2.set_xticks(x)
ax2.set_xticklabels(SPLIT_LABELS, color="#ccc")
ax2.set_ylabel("MAPE (%)", color="#ccc")
ax2.set_title("Perbandingan MAPE: BiLSTM Baseline vs BiLSTM + GA", color="#ddd", fontsize=11)
ax2.legend(fontsize=9, labelcolor="#ccc", facecolor="#1a1a2e", edgecolor="#444")
for sp in ax2.spines.values():
    sp.set_color("#444")

for bar in bars1:
    ax2.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
        f"{bar.get_height():.1f}%", ha="center", va="bottom", color="#aaa", fontsize=8,
    )
for bar in bars2:
    ax2.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
        f"{bar.get_height():.1f}%", ha="center", va="bottom", color="#aaa", fontsize=8,
    )

plt.tight_layout()
fig_cmp_path = PATHS["figures"] / "nb11_bilstm_ga_vs_baseline.png"
fig2.savefig(fig_cmp_path, dpi=160, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()
print(f"✅ Figur perbandingan MAPE tersimpan: {fig_cmp_path}")

## 💾 6. Simpan Artefak Notebook 11

In [ ]:
# ── Simpan trial log, metrik, prediksi, dan history ──────────
trial_df = pd.DataFrame(all_trial_rows)
trials_csv_path = PATHS["ga_trials"] / "nb11_bilstm_ga_trials.csv"
trials_json_path = PATHS["ga_trials"] / "nb11_bilstm_ga_trials.json"
metrics_csv_path = PATHS["metrics"] / "nb11_bilstm_ga_best_metrics.csv"
metrics_json_path = PATHS["metrics"] / "nb11_bilstm_ga_best_metrics.json"
pred_npz_path = PATHS["predictions"] / "nb11_bilstm_ga_predictions.npz"
history_json_path = PATHS["logs"] / "nb11_training_history.json"

trial_df.to_csv(trials_csv_path, index=False)
trial_df.to_json(trials_json_path, orient="records", indent=2, force_ascii=False)

result_rows = []
for label in SPLIT_LABELS:
    m = best_metrics[label]
    nb10 = NB10_BASELINE[label]
    result_rows.append({
        "split": label,
        "MAE_test": m["MAE"],
        "RMSE_test": m["RMSE"],
        "MAPE_test": m["MAPE"],
        "R2_test": m["R2"],
        "NB10_MAPE": nb10["MAPE"],
        "delta_MAPE": nb10["MAPE"] - m["MAPE"],
        "improved": m["MAPE"] < nb10["MAPE"],
        **{f"best_{k}": v for k, v in best_configs[label].items()},
    })

result_df = pd.DataFrame(result_rows)
result_df.to_csv(metrics_csv_path, index=False)

payload = {
    "notebook": "11 - BiLSTM + GA All Splits",
    "timestamp": __import__("datetime").datetime.now().isoformat(timespec="seconds"),
    "ga_config": {
        "POP_SIZE": POP_SIZE,
        "N_GENERATIONS": N_GENERATIONS,
        "ELITE_SIZE": ELITE_SIZE,
        "TOURNAMENT_K": TOURNAMENT_K,
        "CROSSOVER_RATE": CROSSOVER_RATE,
        "MUTATION_RATE": MUTATION_RATE,
        "FIXED_EPOCHS": FIXED_EPOCHS,
        "FINAL_RETRAIN_EPOCHS": FINAL_RETRAIN_EPOCHS,
    },
    "search_space": SEARCH_SPACE_BILSTM,
    "results": result_df.to_dict(orient="records"),
}
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

npz_kwargs = {}
for label in SPLIT_LABELS:
    key = label.replace(":", "_")
    npz_kwargs[f"y_true_{key}"] = best_preds[label]["y_true"]
    npz_kwargs[f"y_pred_{key}"] = best_preds[label]["y_pred"]
np.savez(pred_npz_path, **npz_kwargs)

with open(history_json_path, "w", encoding="utf-8") as f:
    json.dump(best_history, f, indent=2, ensure_ascii=False)

display(result_df)
print("=" * 72)
print("  SIMPAN ARTEFAK — NOTEBOOK 11")
print("=" * 72)
print(f"  ✅ Trial CSV      : {trials_csv_path}")
print(f"  ✅ Trial JSON     : {trials_json_path}")
print(f"  ✅ Metrics CSV    : {metrics_csv_path}")
print(f"  ✅ Metrics JSON   : {metrics_json_path}")
print(f"  ✅ Predictions NPZ: {pred_npz_path}")
print(f"  ✅ History JSON   : {history_json_path}")
print("=" * 72)

## ✅ 7. Checklist Akhir Notebook 11

In [ ]:
# ── Checklist akhir Notebook 11 ──────────────────────────────
trials_done = len(all_trial_rows) > 0
configs_done = len(best_configs) == 4
mape_ok = all("MAPE" in best_metrics[l] for l in SPLIT_LABELS)
mae_ok = all("MAE" in best_metrics[l] for l in SPLIT_LABELS)
rmse_ok = all("RMSE" in best_metrics[l] for l in SPLIT_LABELS)
r2_ok = all("R2" in best_metrics[l] for l in SPLIT_LABELS)
artifacts_ok = all([
    trials_csv_path.exists(),
    trials_json_path.exists(),
    metrics_csv_path.exists(),
    metrics_json_path.exists(),
    pred_npz_path.exists(),
    fig_pred_path.exists(),
    fig_cmp_path.exists(),
])
any_improved = any(
    best_metrics[l]["MAPE"] < NB10_BASELINE[l]["MAPE"]
    for l in SPLIT_LABELS
)

checklist = [
    ("Search space GA BiLSTM terdokumentasi", True),
    ("Trial GA per split tersimpan", trials_done),
    ("Kandidat terbaik per split tersimpan", configs_done),
    ("MAPE per split tersedia", mape_ok),
    ("MAE per split tersedia", mae_ok),
    ("RMSE per split tersedia", rmse_ok),
    ("R² per split tersedia", r2_ok),
    ("Artefak metrik, prediksi, dan figur tersimpan", artifacts_ok),
    ("Setidaknya 1 split lebih baik dari NB10", any_improved),
]

print("=" * 60)
print("  CHECKLIST NOTEBOOK 11")
print("=" * 60)
all_pass = True
for desc, status in checklist:
    icon = "✅" if status else "❌"
    if not status:
        all_pass = False
    print(f"  {icon}  {desc}")
print("=" * 60)

print("\n  📊 Perbandingan Final BiLSTM Baseline vs BiLSTM+GA:")
print(f"  {'Split':<8} {'NB10 MAPE':>10} {'NB11 MAPE':>10} {'Delta':>9} {'Status':>10}")
print("  " + "-" * 52)
for l in SPLIT_LABELS:
    nb10m = NB10_BASELINE[l]["MAPE"]
    nb11m = best_metrics[l]["MAPE"]
    delta = nb10m - nb11m
    status = "✅ BETTER" if delta > 0 else "❌ WORSE"
    print(f"  {l:<8} {nb10m:>9.3f}% {nb11m:>9.3f}% {delta:>+8.3f}% {status}")
print("=" * 60)

if all_pass:
    print("  ✅ SEMUA CHECKLIST LULUS — Notebook 11 selesai!")
    print("     Lanjutkan ke Notebook 12 — CNN-BiLSTM Baseline All Splits.")
else:
    print("  ❌ Ada checklist yang gagal — periksa output di atas.")
print("=" * 60)

---

## 🔗 Navigasi Pipeline

| | Notebook |
|--|----------|
| ← | **[10 - BiLSTM Baseline All Splits](./10%20-%20BiLSTM%20Baseline%20All%20Splits.ipynb)** |
| **→** | **[12 - CNN-BiLSTM Baseline All Splits](./12%20-%20CNN-BiLSTM%20Baseline%20All%20Splits.ipynb)** |

---

<div style="text-align: center; color: #888; font-size: 0.85em; padding: 12px 0;">
Notebook 11 — BiLSTM + GA All Splits &nbsp;|&nbsp;
CNN–BiLSTM + GA Sales Forecasting &nbsp;|&nbsp;
Departemen Sistem Informasi ITS Surabaya &nbsp;|&nbsp; 2025
</div>